# 06_audio_features

DML: bronze_audio_features — Raw audio feature metrics per track.

In [ ]:
%run ../../tools/config/settings

In [ ]:
dbutils.widgets.text("run_id",         "")
dbutils.widgets.text("ingestion_date", "")
run_id         = dbutils.widgets.get("run_id")
ingestion_date = dbutils.widgets.get("ingestion_date")

In [ ]:
from pyspark.sql import functions as F

raw_path = raw_base_path("audio_features", ingestion_date, run_id)
raw_df   = spark.read.json(f"{raw_path}/*.json")

bronze = (
    raw_df
    .select(F.explode("audio_features").alias("af"))
    .filter(F.col("af").isNotNull())
    .select(
        F.col("af.id").alias("track_id"),
        F.col("af.danceability").alias("danceability"),
        F.col("af.energy").alias("energy"),
        F.col("af.key").alias("key"),
        F.col("af.loudness").alias("loudness"),
        F.col("af.mode").alias("mode"),
        F.col("af.speechiness").alias("speechiness"),
        F.col("af.acousticness").alias("acousticness"),
        F.col("af.instrumentalness").alias("instrumentalness"),
        F.col("af.liveness").alias("liveness"),
        F.col("af.valence").alias("valence"),
        F.col("af.tempo").alias("tempo"),
        F.col("af.duration_ms").alias("duration_ms"),
        F.col("af.time_signature").alias("time_signature"),
        F.to_json(F.col("af")).alias("_raw"),
    )
    .withColumn("run_id",         F.lit(run_id))
    .withColumn("ingestion_date", F.to_date(F.lit(ingestion_date)))
)

bronze.write.format("delta").mode("append").saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_audio_features")
print(f"bronze_audio_features: {bronze.count()} rows written")